In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo30"
ESQUEMA = "wanderbricks"

bookings_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_bronce")
    .dropDuplicates(["booking_id"])
    .filter(F.col("total_amount").isNotNull())
    .withColumn("total_amount", F.col("total_amount").cast("decimal(10,2)")))
bookings_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bookings_plata")

payments_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.payments_bronce")
    .dropDuplicates(["payment_id"])
    .filter(F.col("amount").isNotNull()))
payments_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.payments_plata")

reviews_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.reviews_bronce")
    .filter(F.col("is_deleted") == False)
    .dropDuplicates(["review_id"]))
reviews_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.reviews_plata")

clickstream_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_bronce")
    .withColumn("device", F.col("metadata.device"))
    .withColumn("referrer", F.col("metadata.referrer"))
    .drop("metadata"))
clickstream_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.clickstream_plata")

print("✅ Capa plata reconstruida: bookings, payments, reviews, clickstream")